# Harry Potter Text Generation Using GRU - PyTorch Version
This notebook contains the PyTorch implementation of a GRU (Gated Recurrent Unit) model for text generation.
The model is trained on Harry Potter books and learns to predict the next word in a sequence.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm import tqdm
import time
import pickle
import json

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Load the Harry Potter dataset
txt = pd.read_csv('/content/harry_potter_books.txt')

print(f"Dataset shape: {txt.shape}")
print(f"\nFirst few samples:")
print(txt.head(10))
print(f"\nTotal texts: {len(txt)}")

In [ ]:
# Extract text data
X = txt['text']
print(f"Total text entries: {len(X)}")
print(f"\nSample text: {X.iloc[0][:200]}...")

In [ ]:
# Create a custom Tokenizer class
class HarryPotterTokenizer:
    def __init__(self):
        self.word_index = {}
        self.index_word = {}
        self.vocab_size = 0
    
    def fit_on_texts(self, texts):
        """Build vocabulary from texts"""
        self.word_index = {}  # Start from 0 for padding
        index = 1
        
        for text in texts:
            if pd.isna(text):  # Skip NaN values
                continue
            words = str(text).lower().split()
            for word in words:
                if word not in self.word_index:
                    self.word_index[word] = index
                    index += 1
        
        # Create reverse mapping
        self.index_word = {idx: word for word, idx in self.word_index.items()}
        self.vocab_size = len(self.word_index) + 1  # +1 for padding
    
    def texts_to_sequences(self, texts):
        """Convert texts to sequences of integers"""
        sequences = []
        for text in texts:
            if pd.isna(text):
                sequences.append([])
                continue
            words = str(text).lower().split()
            seq = [self.word_index.get(word, 0) for word in words]  # 0 for unknown words
            sequences.append(seq)
        return sequences
    
    def sequences_to_text(self, sequence):
        """Convert sequence of integers back to text"""
        words = []
        for idx in sequence:
            if idx in self.index_word:
                words.append(self.index_word[idx])
        return ' '.join(words)


# Initialize and fit tokenizer
tokenizer = HarryPotterTokenizer()
tokenizer.fit_on_texts(X)

print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Unique words: {len(tokenizer.word_index)}")
print(f"Sample word_index entries: {dict(list(tokenizer.word_index.items())[:10])}")

In [ ]:
# Convert texts to sequences
input_sequences = []
for text in X:
    if pd.isna(text):
        continue
    
    tokenized_sentence = tokenizer.texts_to_sequences([text])[0]
    
    # Create sequences of increasing length (n-gram approach)
    for i in range(1, len(tokenized_sentence)):
        input_sequences.append(tokenized_sentence[:i+1])

print(f"Total input sequences created: {len(input_sequences)}")
if len(input_sequences) > 0:
    print(f"\nSample sequences:")
    for i in range(min(3, len(input_sequences))):
        print(f"Sequence {i}: {input_sequences[i]}")

In [ ]:
# Calculate maximum sequence length
max_len = max([len(x) for x in input_sequences]) if len(input_sequences) > 0 else 0
print(f"Maximum sequence length: {max_len}")

In [ ]:
# Pad sequences to same length
def pad_sequences_custom(sequences, maxlen=None, padding='pre', pad_value=0):
    """Pad sequences to same length"""
    if maxlen is None:
        maxlen = max([len(seq) for seq in sequences]) if sequences else 0
    
    padded = []
    for seq in sequences:
        if padding == 'pre':
            padded_seq = [pad_value] * (maxlen - len(seq)) + seq
        else:  # 'post'
            padded_seq = seq + [pad_value] * (maxlen - len(seq))
        padded.append(padded_seq)
    
    return np.array(padded), maxlen


# Pad the sequences
padded_input_sequences, actual_max_len = pad_sequences_custom(input_sequences, padding='pre')
print(f"Padded sequences shape: {padded_input_sequences.shape}")
print(f"Actual max length: {actual_max_len}")

In [ ]:
# Split into X and y
X_data = padded_input_sequences[:, :-1]  # All tokens except last
y_data = padded_input_sequences[:, -1]   # Last token (target)

print(f"X shape: {X_data.shape}")
print(f"y shape: {y_data.shape}")
print(f"\nSample X: {X_data[0]}")
print(f"Sample y (target): {y_data[0]}")

In [ ]:
# Convert to PyTorch tensors
X_tensor = torch.from_numpy(X_data).long().to(device)
y_tensor = torch.from_numpy(y_data).long().to(device)

print(f"X tensor shape: {X_tensor.shape}")
print(f"y tensor shape: {y_tensor.shape}")

# Create dataset and dataloader
batch_size = 32
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Number of batches: {len(dataloader)}")

In [ ]:
# Define the GRU model for text generation
class HarryPotterGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=1):
        super(HarryPotterGRU, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # GRU layer
        self.gru = nn.GRU(
            embedding_dim,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.5 if num_layers > 1 else 0
        )
        
        # Dense output layer
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        # Embedding
        x = self.embedding(x)
        
        # GRU
        gru_out, hidden = self.gru(x)
        
        # Use the last output for prediction
        last_output = gru_out[:, -1, :]
        
        # Output layer
        output = self.fc(last_output)
        
        return output


# Initialize model
vocab_size = tokenizer.vocab_size
embedding_dim = 128
hidden_size = 150
num_layers = 1

model = HarryPotterGRU(vocab_size, embedding_dim, hidden_size, num_layers)
model = model.to(device)

print(f"Model initialized:")
print(model)

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam")

In [ ]:
# Training function
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for X_batch, y_batch in tqdm(dataloader, desc="Training"):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    
    return avg_loss, accuracy


print("Training function defined!")

In [ ]:
# Train the model
num_epochs = 10
history = {
    'loss': [],
    'accuracy': []
}

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    
    train_loss, train_acc = train_epoch(model, dataloader, criterion, optimizer, device)
    
    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    
    print(f"Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")

print("\nTraining completed!")

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['loss'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['accuracy'])
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training Accuracy')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Function to generate text
def generate_text(seed_text, model, tokenizer, num_words=10, max_seq_length=38):
    """
    Generate text by predicting next words given seed text
    
    Args:
        seed_text: Starting text
        model: Trained GRU model
        tokenizer: Tokenizer instance
        num_words: Number of words to generate
        max_seq_length: Maximum sequence length for padding
    """
    generated_text = seed_text
    
    model.eval()
    with torch.no_grad():
        for _ in range(num_words):
            # Tokenize current text
            token_text = tokenizer.texts_to_sequences([generated_text])[0]
            
            # Pad the sequence
            if len(token_text) > max_seq_length:
                token_text = token_text[-max_seq_length:]
            
            padded_token_text = [0] * (max_seq_length - len(token_text)) + token_text
            
            # Convert to tensor
            input_tensor = torch.tensor([padded_token_text]).long().to(device)
            
            # Predict next word
            output = model(input_tensor)
            predicted_idx = torch.argmax(output, dim=1).item()
            
            # Get the predicted word
            if predicted_idx in tokenizer.index_word:
                predicted_word = tokenizer.index_word[predicted_idx]
                generated_text += " " + predicted_word
                print(generated_text)
                time.sleep(2)  # Delay for readability
            else:
                break  # Stop if invalid index
    
    return generated_text


print("Text generation function defined!")

In [ ]:
# Test the model with different seed texts
test_seeds = [
    "harry",
    "the boy",
    "magic is",
    "dumbledore"
]

print("Testing text generation:\n")
for seed in test_seeds:
    print(f"\n{'='*60}")
    print(f"Seed text: '{seed}'")
    print(f"{'='*60}")
    print(f"Starting: {seed}")
    result = generate_text(seed, model, tokenizer, num_words=10, max_seq_length=38)

In [ ]:
# Save the model
model_save_path = 'harry_potter_gru_pytorch.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

# Save the full model
model_full_path = 'harry_potter_gru_pytorch_full.pt'
torch.save(model, model_full_path)
print(f"Full model saved to {model_full_path}")

# Save tokenizer
tokenizer_save_path = 'harry_potter_tokenizer.pkl'
with open(tokenizer_save_path, 'wb') as f:
    pickle.dump(tokenizer, f)
print(f"Tokenizer saved to {tokenizer_save_path}")

# Save configuration
config = {
    'vocab_size': vocab_size,
    'embedding_dim': embedding_dim,
    'hidden_size': hidden_size,
    'num_layers': num_layers,
    'max_seq_length': actual_max_len
}
config_save_path = 'harry_potter_config.json'
with open(config_save_path, 'w') as f:
    json.dump(config, f)
print(f"Config saved to {config_save_path}")

In [ ]:
# Function to load and use the saved model for inference
def load_model_for_inference(model_path, tokenizer_path, config_path, device):
    """
    Load saved GRU model for inference
    """
    # Load configuration
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    # Load tokenizer
    with open(tokenizer_path, 'rb') as f:
        tokenizer = pickle.load(f)
    
    # Initialize and load model
    model = HarryPotterGRU(
        config['vocab_size'],
        config['embedding_dim'],
        config['hidden_size'],
        config['num_layers']
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    
    return model, tokenizer, config


print("Model loading function defined!")
print("\nYou can load the saved model using:")
print(f"model, tokenizer, config = load_model_for_inference('{model_save_path}', '{tokenizer_save_path}', '{config_save_path}', device)")

In [ ]:
# Additional: Compare GRU vs LSTM for understanding
print("""\n=== GRU vs LSTM Comparison ===")
print("""
GRU (Gated Recurrent Unit):
- Simpler architecture with 2 gates (reset and update)
- Fewer parameters than LSTM
- Faster training
- Good for smaller datasets
- Used in this model for Harry Potter text generation

LSTM (Long Short-Term Memory):
- More complex with 3 gates (input, forget, output)
- More parameters
- Better at capturing long-term dependencies
- Generally better for larger datasets
- Used in the Next Word Predictor model

Both are types of RNNs that solve the vanishing gradient problem!
""")